# Clase 4 — Fundamentos de Vision por Computadora
## Pregunta central
> **¿Cómo puede una computadora aprender a reconocer dígitos escritos a mano?**
## Objetivos de aprendizaje
Al finalizar esta clase vas a poder:
- Cargar y explorar el dataset **Digits** (imágenes de 8×8 píxeles).
- Entender qué hace una **neurona artificial** con datos reales.
- Entrenar una red para reconocer **un solo dígito**, luego **dos**, y finalmente **todos los del 0 al 9**.
- Comparar el mismo modelo en **PyTorch** y **Keras**.
- Modificar la **capacidad** de la red y observar el efecto.

<img src="https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png" width="600"/>

*Ejemplos del dataset MNIST, el hermano mayor de Digits. Nuestro dataset tiene el mismo espíritu, pero en 8×8 píxeles.*

**Conexión con el programa:** Deep Learning de ambos tracks; PyTorch será el framework operativo principal.

## Índice de la clase

Hacé click para saltar a cada sección:

| Paso | Tema |
|---|---|
| [Paso 0](#paso0) | Preparación y primer contacto con los datos |
| [Paso 1](#paso1) | El desafío más simple: ¿es un 0 o no? |
| [Paso 2](#paso2) | Entrenar una neurona (logística) con scikit-learn |
| [Paso 3](#paso3) | Probá la neurona con imágenes reales |
| [Paso 4](#paso4) | Escalando: 0 vs 1 vs 2 |
| [Paso 5](#paso5) | Los conceptos clave de una red neuronal |
| [Paso 6](#paso6) | Clasificación completa (0-9) con PyTorch |
| [Paso 7](#paso7) | La misma red, en Keras |
| [Paso 8](#paso8) | Actividad: ¿más capacidad = mejor? |
| [Paso 9](#paso9) | Evaluación final en test |
| [Síntesis](#sintesis) | Resumen y conceptos |
| [Glosario](#glosario) | Términos clave |

### ¿Cómo usar esta clase?

1. **Ejecutá las celdas en orden** (Shift+Enter) de arriba hacia abajo.
2. Las celdas con `# TODO` están pensadas para que **vos las modifiques** y experimentes.
3. Usamos una **semilla fija** (`SEED = 42`): los resultados son reproducibles en cada ejecución.
4. Si algo se ve raro, **reiniciá el kernel** (Kernel → Restart) y volvé a ejecutar todo.


========================================================================================================

##### **PREPARAR ENTORNO:** 

**1. Clonar repositorio desde github:**
   
  - git clone https://github.com/TataInti/IA_para_Programadores

**2. Dejar listo entorno local:**
   - pip install requirements.txt

**3. Verificar que tienen carpetas 'assets' y elementos en el interior**

```text
assets
      |__audio  
          |__ curso_ia_es.wav
      |__geospatial
          |__ escena_multibanda.tif
          |__ parcela.geojson
      |__images
          |__ china.jpg
          |__ escena_bus_peatones.jpg
          |__ flower.jpg
          |__ MnistExamples.png
```
========================================================================================================

<a id="paso0"></a>
---
## 🚀 Paso 0 — Preparación y primer contacto con los datos


Antes de hablar de redes neuronales, vamos a **ver** el problema. El dataset `Digits` contiene 1 797 imágenes de dígitos escritos a mano, cada una de 8×8 píxeles en escala de grises.

Nuestro objetivo: que la computadora aprenda a mirar una imagen como esta 👇 y diga "es un 7".

In [ ]:
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

# Semilla para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Cargar el dataset
digits = load_digits()
X = (digits.images / 16.0).astype("float32")  # normalizar 0-16 → 0-1
y = digits.target.astype("int64")

print(f"📦 Total de imágenes: {len(X)}")
print(f"🎨 Formato de cada imagen: {X[0].shape}")
print(f"🔢 Etiquetas posibles: {np.unique(y)}")

# Visualizar algunas muestras de cada dígito
fig, axes = plt.subplots(10, 8, figsize=(12, 15))
for digito in range(10):
    indices = np.where(y == digito)[0][:8]
    for i, idx in enumerate(indices):
        ax = axes[digito, i]
        ax.imshow(X[idx], cmap="gray")
        ax.set_title(f"{digito}", fontsize=10)
        ax.axis("off")
plt.suptitle("Dígitos escritos a mano — 8 muestras por clase (8×8 píxeles)", fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

### 🤔 Preguntas de observación

1. ¿Notás que el mismo dígito se escribe de formas distintas? (por ejemplo, el "1" con y sin base)
2. ¿Cuántos valores distintos tiene cada píxel como máximo? *(Pista: antes de normalizar, el rango era 0-16)*
3. ¿Qué dificultades tendría un programa que intente distinguir un **4** de un **9** solo con reglas fijas?

### 💡 ¿Qué ve la computadora, en realidad?

Para nuestra red **no existe** la imagen de un "7". Solo existe un vector de **64 números** entre 0 y 1.
Acá lo vemos ampliado:


In [ ]:
display(pd.DataFrame(digits.data))

In [ ]:
X[1]

In [ ]:
idx_ejemplo = 7  # TODO: probá con otros índices (1, 2, 3...)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(X[idx_ejemplo], cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_title(f"Dígito real: {y[idx_ejemplo]} — los 64 píxeles como números", fontsize=11)

for i in range(8):
    for j in range(8):
        valor = X[idx_ejemplo, i, j]
        color = "black" if valor < 0.5 else "white"
        ax.text(j, i, f"{valor:.1f}", ha="center", va="center",
                color=color, fontsize=9)

plt.colorbar(im, label="Probabilidad (0 = baja , 1 = alta)")
plt.show()


<a id="paso1"></a>
---
## 🎯 Paso 1 — El desafío más simple: ¿Es un 0 o no?

Antes de reconocer los 10 dígitos, vamos a empezar con un problema **binario**: dada una imagen, ¿es un **0** o no?

Esto nos permite entender qué hace una neurona con un ejemplo concreto.

<img src="https://www.futurespace.es/wp-content/uploads/2021/03/deeprrnn.jpg" width="800"/>

*Modelo de una neurona artificial: recibe entradas (píxeles), las multiplica por pesos, suma un bias, y aplica una activación.*

In [ ]:
# Crear problema binario: ¿es un 0?
y_binario = (y == 0).astype(int)  # 1 si es 0, 0 si es cualquier otro

# Dividir en entrenamiento y prueba
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X.reshape(len(X), -1),  # aplanamos a 64 píxeles
    y_binario,
    test_size=0.2,
    stratify=y_binario,
    random_state=SEED,
)

print(f"Entrenamiento: {X_train_bin.shape[0]} imágenes")
print(f"Prueba: {X_test_bin.shape[0]} imágenes")
print(f"¿Cuántos 0s hay en entrenamiento? {y_train_bin.sum()}")
print(f"¿Cuántos NO-0s hay en entrenamiento? {(y_train_bin == 0).sum()}")

# Mostrar algunos ejemplos de cada clase
fig, axes = plt.subplots(1, 6, figsize=(10, 2.5))
# Mostrar 3 ceros y 3 no-ceros
ceros_idx = np.where(y_train_bin == 1)[0][:3]
no_ceros_idx = np.where(y_train_bin == 0)[0][:3]
for i, idx in enumerate(list(ceros_idx) + list(no_ceros_idx)):
    ax = axes[i]
    ax.imshow(X_train_bin[idx].reshape(8, 8), cmap="gray")
    etiqueta = "ES UN 0" if y_train_bin[idx] == 1 else "NO es 0"
    ax.set_title(etiqueta, fontsize=9, color="green" if y_train_bin[idx] == 1 else "red")
    ax.axis("off")
plt.suptitle("Problema: ¿Es un 0 o no?", fontsize=12)
plt.tight_layout()
plt.show()

<a id="paso2"></a>
---
## Paso 2 — Entrenar una neurona con scikit-learn

Vamos a usar **una sola neurona** (regresión logística) para resolver el problema binario.

No usamos Deep Learning todavía; queremos ver qué puede lograr un modelo **simple** antes de apilar capas.

**Conceptos clave:**
- **Entradas** (`x`): los 64 píxeles de la imagen.
- **Pesos** (`w`): valores que la neurona aprende para cada píxel.
- **Bias** (`b`): un desplazamiento que ayuda a ajustar la frontera de decisión.
- **Activación sigmoide**: convierte el resultado en una probabilidad entre 0 y 1.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Entrenar una neurona logística
neurona = LogisticRegression(max_iter=1000, random_state=SEED)
neurona.fit(X_train_bin, y_train_bin)

# Predicciones
y_pred_bin = neurona.predict(X_test_bin)
acc_bin = accuracy_score(y_test_bin, y_pred_bin)

print(f"✅ Precisión del modelo binario: {acc_bin:.1%}")
print("\n📊 Reporte de clasificación:")
print(classification_report(y_test_bin, y_pred_bin, target_names=["NO 0", "ES 0"]))

# Visualizar los pesos aprendidos (qué píxeles "importan" para detectar un 0)
plt.figure(figsize=(7, 6))
plt.imshow(neurona.coef_.reshape(8, 8), cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Peso del píxel")
plt.title("🔍 Lo que la neurona 'aprendió' para reconocer un 0")
plt.axis("off")
plt.show()

### 💡 ¿Qué significan esos colores?

- Los píxeles **rojos** son los que la neurona usa como evidencia de "esto SI es un 0".
- Los píxeles **azules** son los que la neurona usa como evidencia de "esto NO es un 0".
- Los píxeles **blancos/grises** no aportan mucho a la decisión.

**Observación:** ¡La neurona aprendió una forma similar a un círculo\! Pero esto es puramente estadístico: nadie le dijo "busca un círculo".

<a id="paso3"></a>
---
## Paso 3 — Realizar pruebas

Vamos a ver cómo se comporta la neurona con imágenes concretas del conjunto de prueba.

Podés modificar el índice `idx` para probar diferentes imágenes.

In [ ]:
def probar_neurona(idx=11):
    imagen = X_test_bin[idx].reshape(8, 8)
    probabilidad = neurona.predict_proba([X_test_bin[idx]])[0][1]
    prediccion = "ES UN 0" if probabilidad > 0.5 else "NO es 0"
    respuesta_real = "SÍ" if y_test_bin[idx] == 1 else "NO"

    plt.figure(figsize=(4, 4))
    plt.imshow(imagen, cmap="gray")
    plt.title(
        f"Predicción: {prediccion}\nConfianza: {probabilidad:.1%}",
        color="green" if prediccion == "ES UN 0" else "red",
    )
    plt.axis("off")
    plt.show()

    print(f"🖼️  Imagen #{idx} — ¿Es realmente un 0? Respuesta: {respuesta_real}")

try:
    from ipywidgets import IntSlider, interact

    interact(
        probar_neurona,
        idx=IntSlider(min=0, max=len(X_test_bin) - 1, step=1, value=11,
                      description="Índice:", continuous_update=False),
    )
except ImportError:
    print("⚠️ ipywidgets no está disponible; probando con idx=11:")
    probar_neurona(11)


<a id="paso4"></a>
---
## Paso 4 — Escalando el problema: 0 vs 1 vs 2

Ahora vamos a hacer que la red distinga entre **tres clases**: 0, 1 y 2.

Aquí una sola neurona ya no alcanza: necesitamos **tres neuronas de salida** (una por clase), y usamos **softmax** para convertir los puntajes en probabilidades.

Este es el paso previo a clasificar los 10 dígitos completos.

In [ ]:
# Filtrar solo los dígitos 0, 1 y 2
mascara = np.isin(y, [0, 1, 2])
X_tres = X[mascara]
y_tres = y[mascara]

print(f"Imágenes de 0, 1 y 2: {len(X_tres)}")


In [ ]:

# Dividir
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_tres.reshape(len(X_tres), -1),
    y_tres,
    test_size=0.2,
    stratify=y_tres,
    random_state=SEED,
)


In [ ]:

# Usamos una red neuronal MLP simple con scikit-learn
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import ConfusionMatrixDisplay

mlp_3 = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=SEED,
)
mlp_3.fit(X_train_3, y_train_3)

y_pred_3 = mlp_3.predict(X_test_3)
acc_3 = accuracy_score(y_test_3, y_pred_3)

print(f"✅ Precisión (0 vs 1 vs 2): {acc_3:.1%}")
print("\n📊 Reporte detallado:")
print(classification_report(y_test_3, y_pred_3, labels=[0, 1, 2]))


In [ ]:

# Matriz de confusión visual (el color indica cuántas veces se confundió)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test_3, y_pred_3, labels=[0, 1, 2], cmap="Blues", ax=ax)
plt.title("📊 Matriz de confusión — 0 vs 1 vs 2")
plt.show()


In [ ]:

# Curva de pérdida del MLP (sklearn la guarda automáticamente)
plt.figure(figsize=(8, 4))
plt.plot(mlp_3.loss_curve_, marker="o", color="steelblue")
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("📉 Aprendizaje del MLP (0 vs 1 vs 2)")
plt.grid(alpha=0.25)
plt.show()


### 🧠 Preguntas de reflexión

1. ¿Por qué crees que distinguir **0 vs 1 vs 2** es más difícil que **0 vs el resto**?
2. La matriz de confusión muestra dónde se equivoca el modelo. ¿Qué pares de dígitos se confunden más?
3. ¿Qué crees que pasaría si aumentamos las neuronas de la capa oculta de 16 a 64?

<a id="paso5"></a>
---
## Paso 5 — De una neurona a una red: los conceptos clave

Ahora que ya sentiste el problema en la piel, formalicemos las ideas.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/46/Colored_neural_network.svg" width="400"/>

*Red neuronal feed-forward con 3 capas: entrada (rojo), oculta (azul) y salida (verde).*

### Partes de una red neuronal

| Parte | Qué hace | Análogo con los dígitos |
|---|---|---|
| **Peso** (`w`) | Multiplica cada entrada | Qué tanto importa cada píxel |
| **Bias** (`b`) | Desplaza la combinación | Umbral de activación |
| **Activación** | Introduce no-linealidad | Permite separar formas curvas, no solo rectas |
| **Capa** | Grupo de neuronas | Capa oculta = extractor de patrones; Capa de salida = clasificador |
| **Parámetro** | Valor aprendido (pesos + biases) | Aprendidos automáticamente, no escritos a mano |
| **Hiperparámetro** | Configuración fijada antes | Tamaño de capas, learning rate, épocas |

### Activaciones esenciales

| Activación | Idea | Uso típico |
|---|---|---|
| **ReLU** (Rectified Linear Unit)| Si es negativo → 0; si no, lo deja pasar | Capas ocultas (nuestra elección) |
| **Sigmoid** | Comprime entre 0 y 1 | Salidas binarias |
| **Softmax** | Convierte puntajes en probabilidades que suman 1 | Salida multiclase (nuestros 10 dígitos) |

### Cómo aprende una red (el ciclo)

```text
batch de imágenes + etiquetas reales
         |
         v
  1. Forward pass → predicciones
         |
         v
  2. Loss → ¿qué tan mal estamos?
         |
         v
  3. Backpropagation → ¿quién fue el culpable?
         |
         v
  4. Optimizador → ajustar pesos
         |
         +-----> siguiente batch
```

| Concepto | Significado | Ejemplo concreto |
|---|---|---|
| **Batch** | Cuántas imágenes procesamos antes de ajustar | 64 imágenes |
| **Epoch** | Cuántas veces recorremos TODO el dataset | 10 vueltas |
| **Learning rate** | Qué tan agresivo es cada ajuste | 0.01 = pasos moderados |
| **Loss** | Número que mide el error | Cross-entropy para clasificación |
| **Inferencia** | Usar el modelo entrenado (sin aprender más) | Predecir un dígito nuevo |

##### Gráficos de Funciones de Activación

**1. ReLU (Rectified Linear Unit)**

```python
y
  |
1 |               /
  |              /
0 |---------x-----
  |              \
  |               \
-1|
   -2  -1   0   1   2   x
```

- **Ventajas**:
  - Computacionalmente eficiente
  - Popular en capas ocultas

---

### 2. Sigmoid
```python
y (probabilidad)
   |
1  |                    _
   |                   /
0.5|                  /
   |                 /
0  |________________/
   -2  -1   0   1   2   x
```

**Características**:
- **Comportamiento**:
  - Comprime cualquier entrada a [0, 1]
  - Curva en forma de "S" suave y diferenciable
- **Desventajas**:
  - Problema de gradiente que desaparece (para |x| grande)
  - Sesgo hacia ceros (salida media ≈ 0.5)
- **Uso**: Clasificación binaria

---

### 3. Softmax
```python
Para entrada [x1, x2]:
y (probabilidades)
   |
1  |       ____
   |      /    \
0.5|     /      \
   |    /        \
0  |---x---------x---
    -2  -1   0   1   2   x
```

**Características**:
- **Comportamiento**:
  - Convierte scores a probabilidades
  - Generalización multiclase de sigmoid
- **Propiedades**:
  - Salida siempre es una distribución de probabilidad válida
  - Ideal para clasificación con >2 clases
  - Cada salida es independiente de otras


### 🔬 Nuestra red, en una imagen

Concretamente, la red que acabamos de usar tiene esta forma: 64 entradas → capa oculta → 10 salidas.


In [ ]:
def dibujar_red(n_entrada=64, n_oculta=16, n_salida=10):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_axis_off()

    capas = [n_entrada, n_oculta, n_salida]
    xs = [0.12, 0.55, 0.88]
    etiquetas = ["Entrada\n(64 píxeles)", "Capa oculta\n(16 neuronas)", "Salida\n(10 clases)"]
    colores = ["#e74c3c", "#3498db", "#2ecc71"]

    # Para no dibujar 64 puntos, mostramos hasta 12 por capa
    ys = [np.linspace(0.12, 0.88, min(n, 12)) for n in capas]

    # Conexiones (cada peso es una flecha)
    for i in range(len(capas) - 1):
        for y1 in ys[i]:
            for y2 in ys[i + 1]:
                ax.plot([xs[i], xs[i + 1]], [y1, y2], color="gray", alpha=0.15, lw=0.6)

    for i, (x, yy, color, etiqueta) in enumerate(zip(xs, ys, colores, etiquetas)):
        for y in yy:
            ax.scatter(x, y, s=80, color=color, zorder=5, edgecolors="white", linewidths=0.5)
        ax.text(x, 0.93, f"{capas[i]}", ha="center", fontsize=13, weight="bold")
        ax.text(x, 0.04, etiqueta, ha="center", fontsize=9)

    ax.set_title("Arquitectura de nuestra red: 64 → 16 → 10", fontsize=13, pad=20)
    plt.show()

dibujar_red(64, 16, 10)


<a id="paso6"></a>
---
## ⚡ Paso 6 — Clasificación completa (0-9) con PyTorch

Ahora sí: **todos los dígitos, una red neuronal, un framework real**.

Vamos a usar **PyTorch**, que nos da control total sobre cada paso del entrenamiento.

### Componentes que veremos

| Componente | Responsabilidad |
|---|---|
| `TensorDataset` | Une imágenes y etiquetas |
| `DataLoader` | Crea batches y mezcla los datos |
| `nn.Module` | Define la arquitectura de la red |
| `CrossEntropyLoss` | Mide el error de clasificación |
| `Adam` | Optimizador que ajusta los pesos |
| `model.train()` / `model.eval()` | Modo entrenamiento vs. evaluación |
| `torch.no_grad()` | Desactiva gradientes durante inferencia |

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device("cpu")

# División train / val / test estratificada
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=SEED
)

# Convertir a tensores (aplanamos imágenes 8×8 → vector de 64)
X_train_t = torch.tensor(X_train.reshape(-1, 64), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val.reshape(-1, 64), dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
X_test_t = torch.tensor(X_test.reshape(-1, 64), dtype=torch.float32)

# DataLoader para batches
loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Train: {X_train_t.shape[0]} | Val: {X_val_t.shape[0]} | Test: {X_test_t.shape[0]}")

In [ ]:
# Definición de la red
class RedDigitos(nn.Module):
    def __init__(self, hidden_units=32):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(64, hidden_units),   # capa oculta
            nn.ReLU(),                     # activación no lineal
            nn.Linear(hidden_units, 10),   # capa de salida (10 clases)
        )

    def forward(self, x):
        return self.capas(x)

# Función de entrenamiento
def entrenar_pytorch(hidden_units=32, epochs=10):
    modelo = RedDigitos(hidden_units).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    historial_train = []
    historial_val = []

    for ep in range(epochs):
        # --- Entrenamiento ---
        modelo.train()
        perdida_acum = 0.0
        for xb, yb in loader:
            optimizador.zero_grad()        # 1. limpiar gradientes
            logits = modelo(xb.to(device))  # 2. forward
            loss = loss_fn(logits, yb.to(device))  # 3. error
            loss.backward()                # 4. backpropagation
            optimizador.step()             # 5. actualizar pesos
            perdida_acum += loss.item() * len(xb)
        historial_train.append(perdida_acum / len(loader.dataset))

        # --- Validación (sin gradientes) ---
        modelo.eval()
        with torch.no_grad():
            logits_val = modelo(X_val_t.to(device))
            loss_val = loss_fn(logits_val, y_val_t.to(device)).item()
            historial_val.append(loss_val)

    # Inferencia final en test
    modelo.eval()
    with torch.no_grad():
        logits_test = modelo(X_test_t.to(device))
        pred_test = logits_test.argmax(dim=1).cpu().numpy()

    return modelo, historial_train, historial_val, pred_test

# Entrenar
inicio = time.perf_counter()
modelo_torch, loss_train_torch, loss_val_torch, pred_torch = entrenar_pytorch(hidden_units=32, epochs=10)
tiempo_torch = time.perf_counter() - inicio
parametros_torch = sum(p.numel() for p in modelo_torch.parameters())

# Métricas
acc_torch = accuracy_score(y_test, pred_torch)
resultado_torch = pd.Series({
    "Framework": "PyTorch",
    "Parámetros": parametros_torch,
    "Accuracy test": acc_torch,
    "Tiempo (s)": round(tiempo_torch, 3),
    "Loss final train": round(loss_train_torch[-1], 4),
    "Loss final val": round(loss_val_torch[-1], 4),
})
print("Entrenamiento PyTorch completado")
display(resultado_torch.to_frame("Resultado"))

In [ ]:
# Visualizar la curva de aprendizaje
plt.figure(figsize=(9, 4))
plt.plot(loss_train_torch, marker="o", label="Train")
plt.plot(loss_val_torch, marker="s", label="Validation")
plt.xlabel("Época")
plt.ylabel("Loss (Cross-Entropy)")
plt.title("📉 Evolución de la pérdida — PyTorch")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

# Visualizar algunas predicciones
fig, axes = plt.subplots(2, 6, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), range(12)):
    ax.imshow(X_test[idx], cmap="gray")
    color = "green" if pred_torch[idx] == y_test[idx] else "red"
    ax.set_title(f"Pred: {pred_torch[idx]} | Real: {y_test[idx]}", color=color)
    ax.axis("off")
plt.suptitle("Predicciones de PyTorch en imágenes de test", fontsize=12)
plt.tight_layout()
plt.show()

### 🧬 ¿Qué busca la red en los píxeles?

Así como en el Paso 2 vimos el mapa de pesos de una sola neurona, ahora podemos ver los **patrones 8×8** que aprendió la capa oculta de PyTorch. Cada miniatura es una neurona; combinadas, arman la evidencia para decidir cada dígito.


In [ ]:
# La primera capa aprende "plantillas" de 8×8: cada fila de pesos es una neurona oculta.
w1_torch = modelo_torch.capas[0].weight.detach().cpu().numpy()  # (hidden_units, 64)

n = w1_torch.shape[0]
cols = 8
filas = -(-n // cols)

fig, axes = plt.subplots(filas, cols, figsize=(2.2 * cols, 2.2 * filas))
for i in range(filas * cols):
    ax = axes.ravel()[i]
    if i < n:
        ax.imshow(w1_torch[i].reshape(8, 8), cmap="magma")
        ax.set_title(f"N{i}", fontsize=8)
    ax.axis("off")
plt.suptitle("🧬 Patrones 8×8 aprendidos por la capa oculta (PyTorch)", fontsize=13)
plt.tight_layout()
plt.show()


<a id="paso7"></a>
---
## Paso 7 — La misma red, en Keras (TensorFlow)

Ahora repetimos el **mismo experimento** con Keras, que ofrece una interfaz de más alto nivel: `compile`, `fit`, `predict`.

La idea es mostrar que **los conceptos son idénticos**, solo cambia la forma de escribirlos.

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

# Misma arquitectura: 64 entradas → 32 ocultas → 10 salidas
modelo_keras = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10),  # logits, sin activación (from_logits=True)
])

modelo_keras.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

# Entrenar
inicio = time.perf_counter()
historia_keras = modelo_keras.fit(
    X_train.reshape(-1, 64),
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val.reshape(-1, 64), y_val),
    verbose=0,
)
tiempo_keras = time.perf_counter() - inicio

# Inferencia
logits_keras = modelo_keras.predict(X_test.reshape(-1, 64), verbose=0)
pred_keras = logits_keras.argmax(axis=1)
acc_keras = accuracy_score(y_test, pred_keras)

resultado_keras = pd.Series({
    "Framework": "Keras",
    "Parámetros": modelo_keras.count_params(),
    "Accuracy test": acc_keras,
    "Tiempo (s)": round(tiempo_keras, 3),
    "Loss final train": round(historia_keras.history["loss"][-1], 4),
    "Loss final val": round(historia_keras.history["val_loss"][-1], 4),
})
print("✅ Entrenamiento Keras completado")
display(resultado_keras.to_frame("Resultado"))

In [ ]:
# Comparativa visual: curvas de loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(loss_train_torch, marker="o", label="PyTorch train")
plt.plot(loss_val_torch, marker="s", label="PyTorch val")
plt.plot(historia_keras.history["loss"], marker="^", label="Keras train")
plt.plot(historia_keras.history["val_loss"], marker="v", label="Keras val")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("📉 Pérdida: PyTorch vs Keras")
plt.legend()
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
comparacion = pd.DataFrame([
    ["PyTorch", parametros_torch, acc_torch, round(tiempo_torch, 3)],
    ["Keras", modelo_keras.count_params(), acc_keras, round(tiempo_keras, 3)],
], columns=["Framework", "Parámetros", "Accuracy", "Tiempo (s)"])
bar_width = 0.35
x = np.arange(2)
plt.bar(x - bar_width/2, [acc_torch, acc_keras], bar_width, label="Accuracy")
plt.xticks(x, ["PyTorch", "Keras"])
plt.ylabel("Accuracy en test")
plt.title("🥊 Comparación de frameworks")
plt.ylim(0.8, 1.0)
plt.legend()
plt.grid(alpha=0.25, axis="y")

plt.tight_layout()
plt.show()

display(comparacion.round(4))

### 🧬 Los mismos patrones, pero en Keras

La arquitectura es idéntica a la de PyTorch, así que debería aprender patrones parecidos. Compará visualmente.


In [ ]:
# En Keras, los pesos de la primera capa densa tienen forma (64, 32); transponemos a (32, 64).
w1_keras = modelo_keras.layers[0].get_weights()[0].T

n = w1_keras.shape[0]
cols = 8
filas = -(-n // cols)

fig, axes = plt.subplots(filas, cols, figsize=(2.2 * cols, 2.2 * filas))
for i in range(filas * cols):
    ax = axes.ravel()[i]
    if i < n:
        ax.imshow(w1_keras[i].reshape(8, 8), cmap="Blues")
        ax.set_title(f"N{i}", fontsize=8)
    ax.axis("off")
plt.suptitle("🧬 Patrones 8×8 aprendidos por la capa oculta (Keras)", fontsize=13)
plt.tight_layout()
plt.show()


### 🧐 Preguntas de observación

1. ¿Por qué ambos modelos tienen la **misma cantidad de parámetros**?
2. ¿Los pesos aprendidos deberían ser **idénticos**? ¿Por qué no lo son? *(Pista: la inicialización aleatoria)*
3. ¿La menor loss de **train** garantiza el mejor resultado en **test**? ¿Por qué?
4. Si una curva de validation empieza a subir mientras train sigue bajando, ¿qué está pasando? *(overfitting)*

<a id="paso8"></a>
---
## Paso 8 — Actividad: ¿Más capacidad = mejor resultado?


**Capacidad** es la flexibilidad de la red para representar relaciones complejas. Se controla principalmente con el número de neuronas en la capa oculta.

Vamos a experimentar modificando `HIDDEN_UNITS`. Probá diferentes valores y compará:
Ahora lo hacemos con un **slider** (sin tocar código) y, debajo, un **barrido automático** que compara varios valores de una vez.
- ¿Mejora la **accuracy en validation**?
- ¿Cuántos **parámetros** tiene ahora?
- ¿La **loss** baja más rápido o más lento?

> ⚠️ **Regla de oro:** usá **validation** para comparar modelos. El **test** solo se toca una vez, al final.

In [ ]:
from IPython.display import clear_output

def experimento_capacidad(hidden_units=64):
    global modelo_exp, loss_train_exp, loss_val_exp, pred_exp

    modelo_exp, loss_train_exp, loss_val_exp, pred_exp = entrenar_pytorch(
        hidden_units=hidden_units, epochs=10
    )

    # Métricas de validation (NO test todavía)
    modelo_exp.eval()
    with torch.no_grad():
        logits_val_exp = modelo_exp(X_val_t.to(device))
        pred_val_exp = logits_val_exp.argmax(dim=1).cpu().numpy()

    parametros_exp = sum(p.numel() for p in modelo_exp.parameters())
    acc_val_exp = accuracy_score(y_val, pred_val_exp)

    resultado_exp = pd.Series({
        "hidden_units": hidden_units,
        "parámetros": parametros_exp,
        "loss final train": round(loss_train_exp[-1], 4),
        "loss final val": round(loss_val_exp[-1], 4),
        "accuracy validation": round(acc_val_exp, 4),
    })

    clear_output(wait=True)
    display(resultado_exp.to_frame("Experimento"))

    plt.figure(figsize=(8, 4))
    plt.plot(loss_train_exp, marker="o", label=f"Train ({hidden_units} unidades)")
    plt.plot(loss_val_exp, marker="s", label=f"Val ({hidden_units} unidades)")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(f"📉 Experimento con {hidden_units} neuronas ocultas")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

try:
    from ipywidgets import IntSlider, interact

    interact(
        experimento_capacidad,
        hidden_units=IntSlider(min=8, max=256, step=8, value=64,
                               description="Neuronas ocultas:", continuous_update=False),
    )
except ImportError:
    print("⚠️ ipywidgets no está disponible; usando el valor por defecto (64):")
    experimento_capacidad(64)


In [ ]:
# TODO (opcional): probá otras configuraciones agregando valores a la lista.
configs = [8, 16, 32, 64, 128, 256]
filas_barrido = []

for hu in configs:
    modelo_b, loss_train_b, loss_val_b, _ = entrenar_pytorch(hidden_units=hu, epochs=10)
    modelo_b.eval()
    with torch.no_grad():
        pred_val_b = modelo_b(X_val_t.to(device)).argmax(dim=1).cpu().numpy()

    filas_barrido.append({
        "neuronas ocultas": hu,
        "parámetros": sum(p.numel() for p in modelo_b.parameters()),
        "loss final val": round(loss_val_b[-1], 4),
        "accuracy validation": round(accuracy_score(y_val, pred_val_b), 4),
    })

df_barrido = pd.DataFrame(filas_barrido)
print("Barrido automático de capacidad (medido en validation)")
display(df_barrido)

plt.figure(figsize=(8, 4))
plt.plot(df_barrido["neuronas ocultas"], df_barrido["accuracy validation"],
         marker="o", color="steelblue")
plt.xlabel("Neuronas en la capa oculta")
plt.ylabel("Accuracy en validation")
plt.title("¿Más capacidad siempre es mejor?")
plt.xticks(configs)
plt.grid(alpha=0.25)
plt.show()


<a id="paso9"></a>
---
## Paso 9 — Evaluación final en test


Una vez que elegiste tu mejor modelo comparando en **validation**, evalualo en **test** para estimar cómo se comportaría en datos **completamente nuevos**.

> **Este es el momento de la verdad.**

ℹ**Importante:** acá se evalúa el **último modelo que entrenaste** en el Paso 8 (con el valor final del slider). Si querés evaluar otra configuración, volvé y cambiá el slider.


In [ ]:
# Evaluar el modelo de la actividad en test (SOLO UNA VEZ)
modelo_exp.eval()
with torch.no_grad():
    logits_test_exp = modelo_exp(X_test_t.to(device))
    pred_test_exp = logits_test_exp.argmax(dim=1).cpu().numpy()

acc_test_exp = accuracy_score(y_test, pred_test_exp)
cm = confusion_matrix(y_test, pred_test_exp)

print(f"🎯 Accuracy final en test: {acc_test_exp:.1%}")

print("\n📊 Reporte por dígito:")
print(classification_report(y_test, pred_test_exp, digits=3))

# Visualizar matriz de confusión
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Reds")
plt.colorbar()
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("📊 Matriz de confusión — Evaluación final")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
plt.show()

<a id="sintesis"></a>
---
## Síntesis de la clase

### Lo que vimos

1. **El problema real:** reconocer dígitos escritos a mano a partir de imágenes de 8×8 píxeles.
2. **Desde lo simple a lo complejo:** primero un dígito vs. todos, luego 3 dígitos, finalmente los 10.
3. **Una neurona** puede aprender a detectar patrones visuales (¡sin que nadie le diga qué buscar\!).
4. **Una red neuronal** apila neuronas con activaciones no lineales para resolver problemas más complejos.
5. **Los frameworks** (PyTorch, Keras) automatizan tensores, gradientes y optimización, pero las decisiones clave siguen siendo nuestras.
6. **Más capacidad no siempre es mejor:** hay que validar en datos que el modelo no vio durante entrenamiento.

### Conceptos que deberías poder explicar con tus palabras

| Concepto | ¿Podés explicarlo? |
|---|---|
| Peso y bias | ¿Qué papel juegan en la decisión? |
| ReLU | ¿Por qué no puede ser todo lineal? |
| Softmax | ¿Por qué necesitamos probabilidades que sumen 1? |
| Batch y epoch | ¿Cuál es la diferencia? |
| Train / val / test | ¿Para qué sirve cada uno? |
| Overfitting | ¿Cómo lo detectás en las curvas? |
| PyTorch vs Keras | ¿Qué diferencias de interfaz notaste? |

### Autoevaluación (hacé click para ver las respuestas)

<details>
<summary>💡 ¿Cómo detectás el <b>overfitting</b> en las curvas?</summary>

Si la pérdida de <b>entrenamiento</b> sigue bajando pero la de <b>validación</b> empieza a <b>subir</b> (o se estanca), el modelo está memorizando ruido en lugar de generalizar a datos nuevos.

</details>

<details>
<summary>💡 ¿Por qué usamos <b>softmax</b> y no sigmoide para los 10 dígitos?</summary>

Softmax convierte los puntajes en <b>probabilidades que suman 1</b> entre las 10 clases (una decisión "competitiva"). Sigmoid solo decide entre dos; no fuerza a que todo sume 1.

</details>

<details>
<summary>💡 ¿Cuál es la diferencia entre <b>batch</b> y <b>epoch</b>?</summary>

Un <b>batch</b> es el grupo de imágenes que procesamos antes de actualizar los pesos. Un <b>epoch</b> es una pasada completa por TODO el dataset (muchos batches seguidos).

</details>

<details>
<summary>💡 ¿Para qué sirve cada conjunto: <b>train</b>, <b>validation</b> y <b>test</b>?</summary>

- <b>Train</b>: con él se aprende (se ajustan los pesos).
- <b>Validation</b>: para comparar modelos y elegir hiperparámetros sin tocar test.
- <b>Test</b>: se usa una sola vez, al final, para estimar el desempeño en datos completamente nuevos.

</details>



<a id="glosario"></a>
---
## 📖 Anexo — Glosario mínimo

| Término | Explicación breve |
|---|---|
| **Tensor** | Arreglo numérico con una o más dimensiones |
| **Shape** | Tamaño del tensor en cada dimensión (ej: `(64, 8, 8)`) |
| **Forward pass** | Cálculo que produce una predicción a partir de una entrada |
| **Loss** | Número que cuantifica el error del modelo |
| **Backpropagation** | Cálculo de cómo contribuyó cada parámetro al error |
| **Gradient descent** | Actualización de parámetros para reducir la loss |
| **Batch** | Grupo de muestras procesado en una actualización |
| **Epoch** | Recorrido completo por los datos de entrenamiento |
| **Learning rate** | Tamaño de cada actualización de pesos |
| **Inferencia** | Forward pass sin actualizar parámetros (usar el modelo) |
| **Optimizador** | Regla que actualiza parámetros usando gradientes (ej: Adam) |
| **Logit** | Puntaje de salida antes de convertirlo en probabilidad |
| **Arquitectura** | Organización de capas y conexiones de la red |
| **Framework** | Biblioteca integrada (PyTorch, Keras) para definir, entrenar y ejecutar modelos |
| **Autodiferenciación** | Cálculo automático de gradientes a partir de operaciones |
| **Overfitting** | Cuando el modelo memoriza el entrenamiento pero falla en datos nuevos |
| **Regularización** | Técnicas para evitar overfitting |
| **Capacidad** | Flexibilidad del modelo; controlada por número de parámetros/capas |